# 🚀 DIO vLLM Worker (Colab T4)
This notebook runs a high-performance vLLM worker on Google Colab and connects it to your local DIO Manager.

In [ ]:
# 1. Install Dependencies
%pip install vllm grpcio-tools protobuf pyngrok

In [ ]:
# 2. Define Protobufs (Same as local)
import os
os.makedirs('api/proto', exist_ok=True)

proto_content = """
syntax = "proto3";
package dio;
option go_package = "github.com/nisaral/dio/api/proto";

service Orchestrator {

  rpc RegisterWorker (RegisterRequest) returns (RegisterResponse);
  rpc Infer (InferenceRequest) returns (InferenceResponse);
  rpc ExecuteInference (InferenceRequest) returns (InferenceResponse);
}

service InferenceWorker {
  rpc Predict (InferenceRequest) returns (InferenceResponse);
  rpc CheckHealth (google.protobuf.Empty) returns (google.protobuf.Empty);
}

message RegisterRequest {
  string worker_id = 1;
  string address = 2;
  string tier = 3;
  int64 vram_gb = 4;
}

message RegisterResponse {
  bool success = 1;
  string message = 2;
}

message InferenceRequest {
  string model_id = 1;
  bytes data = 2;
  string tier = 3;
}

message InferenceResponse {
  bytes output = 1;
  double latency_ms = 2;
  double ttft_ms = 3;
  int64 tokens_used = 4;
}

import "google/protobuf/empty.proto";
"""

with open('api/proto/dio.proto', 'w') as f:
    f.write(proto_content)

%python -m grpc_tools.protoc -I. --python_out=. --grpc_python_out=. api/proto/dio.proto

In [ ]:
# 3. Worker Code (vLLM Optimized)
import grpc
import time
import socket
import logging
from concurrent import futures
from vllm import LLM, SamplingParams
import api.proto.dio_pb2 as dio_pb2
import api.proto.dio_pb2_grpc as dio_pb2_grpc

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("vLLM-Worker")

# --- Config ---
MANAGER_ADDRESS = 'https://96e1d967750c.ngrok-free.app' # <--- REPLACE THIS WITH YOUR NGROK ADDRESS
MODEL_ID = "facebook/opt-125m" # Small model for testing, swap for Llama-3-8B if you have High-RAM
WORKER_PORT = "50051"

# Initialize vLLM
logger.info(f"⏳ Loading vLLM Model: {MODEL_ID}...")
llm = LLM(model=MODEL_ID, dtype="float16", gpu_memory_utilization=0.8)
sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=100)
logger.info("✅ Model Loaded!")

class InferenceWorker(dio_pb2_grpc.InferenceWorkerServicer):
    def Predict(self, request, context):
        start_time = time.time()
        prompt = request.data.decode("utf-8")
        
        # vLLM Generation
        outputs = llm.generate([prompt], sampling_params, use_tqdm=False)
        generated_text = outputs[0].outputs[0].text
        
        latency = (time.time() - start_time) * 1000
        tokens = len(outputs[0].outputs[0].token_ids)
        
        logger.info(f"⚡ Request processed: {tokens} tokens in {latency:.2f}ms")
        
        return dio_pb2.InferenceResponse(
            output=generated_text.encode("utf-8"),
            latency_ms=latency,
            ttft_ms=latency / 10.0, # Approx for non-streaming
            tokens_used=tokens
        )

    def CheckHealth(self, request, context):
        return dio_pb2.google_dot_protobuf_dot_empty__pb2.Empty()

def register(public_url):
    # FIX: Handle ngrok secure connection
    if "ngrok" in MANAGER_ADDRESS:
        target = MANAGER_ADDRESS.replace("https://", "").replace("http://", "")
        creds = grpc.ssl_channel_credentials()
        channel = grpc.secure_channel(target, creds)
    else:
        channel = grpc.insecure_channel(MANAGER_ADDRESS)

    stub = dio_pb2_grpc.OrchestratorStub(channel)
    
    # We need to tell the manager how to reach US (the Colab worker)
    # Since Colab is behind NAT, we usually need a reverse tunnel for the manager to call us back.
    # FOR SIMPLICITY in this specific test: We assume the Manager does NOT dial back the worker 
    # for health checks, or we use ngrok on Colab too.
    # Here we just register to announce presence.
    
    try:
        stub.RegisterWorker(dio_pb2.RegisterRequest(
            worker_id="colab-vllm-t4",
            address=public_url, # Send the public ngrok URL
            tier="large",
            vram_gb=16
        ), metadata=[('ngrok-skip-browser-warning', 'true')])
        logger.info("✅ Registered with Manager!")
    except Exception as e:
        logger.error(f"❌ Registration failed: {e}")

def serve():
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    dio_pb2_grpc.add_InferenceWorkerServicer_to_server(InferenceWorker(), server)
    server.add_insecure_port(f'[::]:{WORKER_PORT}')
    server.start()
    logger.info(f"🎧 Worker listening on {WORKER_PORT}")

    # Start ngrok HTTP tunnel (Free tier compatible)
    from pyngrok import ngrok
    # Kill old tunnels
    ngrok.kill()
    public_url = ngrok.connect(WORKER_PORT, "http").public_url
    logger.info(f"🚇 Ngrok Tunnel: {public_url}")

    register(public_url)
    try:
        while True:
            time.sleep(86400)
    except KeyboardInterrupt:
        server.stop(0)

if __name__ == '__main__':
    serve()